# Single Image Prediction
In this notebook, we load our fully trained 4-block `EmotionCNN` and test it on a single random image from our dataset. This is the exact logic we will use for Phase 2 live webcam inference!


In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from PIL import Image
import matplotlib.pyplot as plt
import os
import random
import numpy as np
from pathlib import Path

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is not available for prediction!")

device = torch.device('cuda')
print(f"Prediction Device: {device}")

## 1. Rebuild the 4-Block CNN & Load Model Weights


In [ ]:
class EmotionCNN(nn.Module):
    def __init__(self, num_classes=7):
        super(EmotionCNN, self).__init__()
        
        self.block1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.25)
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.25)
        )
        self.block3 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.25)
        )
        self.block4 = nn.Sequential(
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.25)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512 * 3 * 3, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )
        
    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        return self.classifier(x)

model_path = Path('../models/best_emotion_cnn.pth')
model = EmotionCNN(num_classes=7).to(device)

if model_path.exists():
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    print(f"Loaded best model weights from {model_path}.")
else:
    print("Model weights file not found. Train the model first in train.ipynb!")

## 2. Image Preprocessing Pipeline
The image must be processed as 48x48 Grayscale normalized tensor.


In [ ]:
classes = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']

preprocess = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((48, 48)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

## 3. Predict on a Random Image


In [ ]:
test_dir = Path('../dataset/test')
all_image_paths = list(test_dir.glob('*/*.jpg')) + list(test_dir.glob('*/*.png'))

if len(all_image_paths) > 0:
    random_img_path = random.choice(all_image_paths)
    true_label = random_img_path.parent.name
    
    raw_img = Image.open(random_img_path)
    tensor_img = preprocess(raw_img).unsqueeze(0).to(device)
    
    with torch.no_grad():
        outputs = model(tensor_img)
        probabilities = torch.softmax(outputs, dim=1)
        predicted_idx = torch.argmax(probabilities, dim=1).item()
        predicted_label = classes[predicted_idx]
        confidence = probabilities[0][predicted_idx].item() * 100
        
    plt.figure(figsize=(5, 5))
    plt.imshow(raw_img, cmap='gray')
    plt.title(f"True: {true_label} | Pred: {predicted_label} ({confidence:.1f}%)")
    plt.axis('off')
    plt.show()
else:
    print("No test images found under dataset/test/")